# Chicago Crime Analytics — Python Data Validation

This notebook independently aggregates record-level data from
`public.clean_chicago_crimes` with Pandas and reconciles the results to
the Milestone 6 PostgreSQL analytical views.

**Metric contract:** counts are reported incident records, not
population-normalized crime rates. Annual comparisons use the complete
calendar years 2023–2025. Arrest percentages are not clearance or
conviction rates. Community-area percentage ranks require at least 500
prior-year incidents; absolute ranks include all 77 official areas.

## 1. Environment and secure database connection

Connection values come from environment variables or a local `.env`
file excluded by Git. Credentials and connection URLs are never printed.
PostgreSQL sessions are forced to read-only mode.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import URL, create_engine, text

%matplotlib inline


def find_project_root() -> Path:
    """Find the repository root whether execution starts there or in notebooks/."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / ".env.example").exists() and (candidate / "sql").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing .env.example and sql/.")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env", override=False)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Project root located from the current working directory.")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__} | Matplotlib: {plt.matplotlib.__version__}")

Project root located from the current working directory.
Pandas: 2.3.3 | NumPy: 2.0.2 | Matplotlib: 3.9.4


In [2]:
def build_read_only_engine():
    """Create a read-only PostgreSQL engine without printing credentials."""
    host = os.getenv("POSTGRES_HOST") or None
    user = os.getenv("POSTGRES_USER") or None
    password = os.getenv("POSTGRES_PASSWORD") or None
    database = os.getenv("POSTGRES_DB", "chicago_crime")
    port = int(os.getenv("POSTGRES_PORT", "5432")) if host else None

    url = URL.create(
        "postgresql+psycopg",
        username=user,
        password=password,
        host=host,
        port=port,
        database=database,
    )
    connect_args = {"options": "-c default_transaction_read_only=on"}
    sslmode = os.getenv("POSTGRES_SSLMODE")
    if host and sslmode:
        connect_args["sslmode"] = sslmode

    engine = create_engine(url, future=True, connect_args=connect_args)
    return engine, database, host


engine, database_name, database_host = build_read_only_engine()
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text("""
            SELECT current_database() AS database_name,
                   current_setting('transaction_read_only') AS transaction_read_only
        """),
        connection,
    )

assert connection_check.loc[0, "transaction_read_only"] == "on"
print(
    f"Connected securely to {database_name} via "
    f"{'host ' + database_host if database_host else 'local PostgreSQL socket'}; "
    "transactions are read-only."
)
display(connection_check)

Connected securely to chicago_crime via local PostgreSQL socket; transactions are read-only.


,database_name,transaction_read_only
0,chicago_crime,on


## 2. Load the canonical record-level population

The ordered query loads only fields needed for validation. No analytical
view is used to construct the Pandas results below.

In [3]:
BASE_QUERY = text("""
    SELECT source_id,
           crime_date,
           crime_year,
           crime_month,
           month_name,
           season,
           day_of_week_num,
           day_of_week,
           hour_of_day,
           primary_type,
           arrest,
           domestic,
           community_area,
           community_area_eligible_flag
    FROM public.clean_chicago_crimes
    ORDER BY source_id
""")

records = pd.read_sql_query(BASE_QUERY, engine, parse_dates=["crime_date"])

integer_columns = [
    "source_id", "crime_year", "crime_month", "day_of_week_num", "hour_of_day"
]
for column in integer_columns:
    records[column] = pd.to_numeric(records[column], errors="raise")

print(f"Loaded {len(records):,} clean records with {records['source_id'].nunique():,} unique source IDs.")
print(f"Date coverage: {records['crime_date'].min().date()} through {records['crime_date'].max().date()}")
display(records.head(3))

Loaded 761,563 clean records with 761,563 unique source IDs.
Date coverage: 2023-01-01 through 2025-12-31


,source_id,crime_date,crime_year,crime_month,month_name,season,day_of_week_num,day_of_week,hour_of_day,primary_type,arrest,domestic,community_area,community_area_eligible_flag
0,27279,2023-01-01,2023,1,January,Winter,7,Sunday,5,HOMICIDE,True,False,24.0000,True
1,27280,2023-01-01,2023,1,January,Winter,7,Sunday,4,HOMICIDE,True,False,60.0000,True
2,27281,2023-01-01,2023,1,January,Winter,7,Sunday,9,HOMICIDE,True,False,49.0000,True


In [4]:
sql_kpis = pd.read_sql_query(text("SELECT * FROM public.vw_executive_kpis"), engine)
sql_yearly = pd.read_sql_query(
    text("SELECT * FROM public.vw_yearly_crime_trends ORDER BY crime_year"), engine
)
sql_community = pd.read_sql_query(
    text("""
        SELECT *
        FROM public.vw_community_area_yoy
        ORDER BY current_year, community_area
    """),
    engine,
)
community_lookup = pd.read_sql_query(
    text("SELECT * FROM public.vw_community_area_lookup ORDER BY community_area"), engine
)

executive_validation = pd.DataFrame({
    "metric": [
        "record_count", "distinct_source_ids", "first_crime_date", "last_crime_date",
        "community_area_eligible_count"
    ],
    "python_value": [
        len(records),
        records["source_id"].nunique(),
        str(records["crime_date"].min().date()),
        str(records["crime_date"].max().date()),
        int(records["community_area_eligible_flag"].sum()),
    ],
    "sql_value": [
        int(sql_kpis.loc[0, "reported_incident_count"]),
        int(sql_kpis.loc[0, "reported_incident_count"]),
        str(sql_kpis.loc[0, "first_crime_date"]),
        str(sql_kpis.loc[0, "last_crime_date"]),
        int(sql_kpis.loc[0, "community_area_eligible_count"]),
    ],
})
executive_validation["matches"] = (
    executive_validation["python_value"].astype(str)
    == executive_validation["sql_value"].astype(str)
)
display(executive_validation)

,metric,python_value,sql_value,matches
0,record_count,761563,761563,True
1,distinct_source_ids,761563,761563,True
2,first_crime_date,2023-01-01,2023-01-01,True
3,last_crime_date,2025-12-31,2025-12-31,True
4,community_area_eligible_count,760496,760496,True


## 3. Independent annual totals and year-over-year changes

Pandas groups the record-level table by year, then applies the documented
formula `(current - previous) / previous * 100`. SQL values are loaded
only after the independent calculation is complete.

In [5]:
py_yearly = (
    records.groupby("crime_year", as_index=False)
    .size()
    .rename(columns={"size": "python_incident_count"})
    .sort_values("crime_year")
    .reset_index(drop=True)
)
py_yearly["python_previous_count"] = py_yearly["python_incident_count"].shift(1)
py_yearly["python_absolute_change"] = (
    py_yearly["python_incident_count"] - py_yearly["python_previous_count"]
)
py_yearly["python_percentage_change"] = np.where(
    py_yearly["python_previous_count"].eq(0),
    np.nan,
    100.0 * py_yearly["python_absolute_change"] / py_yearly["python_previous_count"],
)

annual_validation = py_yearly.merge(
    sql_yearly[[
        "crime_year", "reported_incident_count", "previous_year_incident_count",
        "absolute_change", "percentage_change"
    ]],
    on="crime_year",
    how="outer",
    validate="one_to_one",
)
numeric_sql_columns = [
    "reported_incident_count", "previous_year_incident_count",
    "absolute_change", "percentage_change"
]
for column in numeric_sql_columns:
    annual_validation[column] = pd.to_numeric(annual_validation[column], errors="coerce")

annual_validation["count_difference"] = (
    annual_validation["python_incident_count"]
    - annual_validation["reported_incident_count"]
)
annual_validation["yoy_percentage_difference"] = (
    annual_validation["python_percentage_change"]
    - annual_validation["percentage_change"]
)
display(annual_validation.round(10))

,crime_year,python_incident_count,python_previous_count,python_absolute_change,python_percentage_change,reported_incident_count,previous_year_incident_count,absolute_change,percentage_change,count_difference,yoy_percentage_difference
0,2023,263844,NaN,NaN,NaN,263844,NaN,NaN,NaN,0,NaN
1,2024,259633,"263,844.0000","-4,211.0000",-1.5960,259633,"263,844.0000","-4,211.0000",-1.5960,0,0.0000
2,2025,238086,"259,633.0000","-21,547.0000",-8.2990,238086,"259,633.0000","-21,547.0000",-8.2990,0,0.0000


## 4. Independent community-area comparisons and ranks

Pandas builds a complete year-by-area grid from the 77-area lookup,
aggregates valid source community-area identifiers, calculates adjacent
changes, and reproduces both ranking methods. Missing or invalid area
identifiers remain in citywide validation but are excluded here.

In [6]:
years = np.sort(records["crime_year"].unique())
grid = pd.MultiIndex.from_product(
    [years, community_lookup["community_area"].astype(int)],
    names=["crime_year", "community_area"],
).to_frame(index=False)

eligible_records = records.loc[records["community_area_eligible_flag"]].copy()
py_area_counts = (
    eligible_records.groupby(["crime_year", "community_area"], as_index=False)
    .size()
    .rename(columns={"size": "current_year_incident_count"})
)
py_area = (
    grid.merge(py_area_counts, on=["crime_year", "community_area"], how="left")
    .merge(community_lookup, on="community_area", how="left", validate="many_to_one")
    .sort_values(["community_area", "crime_year"])
    .reset_index(drop=True)
)
py_area["current_year_incident_count"] = (
    py_area["current_year_incident_count"].fillna(0).astype("int64")
)
py_area["previous_year"] = py_area.groupby("community_area")["crime_year"].shift(1)
py_area["previous_year_incident_count"] = (
    py_area.groupby("community_area")["current_year_incident_count"].shift(1)
)
py_area = py_area.loc[py_area["previous_year"].notna()].copy()
py_area = py_area.rename(columns={"crime_year": "current_year"})
py_area["previous_year"] = py_area["previous_year"].astype(int)
py_area["absolute_change"] = (
    py_area["current_year_incident_count"] - py_area["previous_year_incident_count"]
)
py_area["percentage_change"] = np.where(
    py_area["previous_year_incident_count"].eq(0),
    np.nan,
    100.0 * py_area["absolute_change"] / py_area["previous_year_incident_count"],
)
py_area["percentage_rank_eligible"] = py_area["previous_year_incident_count"].ge(500)
py_area["absolute_change_rank"] = (
    py_area.groupby("current_year")["absolute_change"]
    .rank(method="min", ascending=False)
)
py_area["absolute_decrease_rank"] = (
    py_area.groupby("current_year")["absolute_change"]
    .rank(method="min", ascending=True)
)
eligible_mask = py_area["percentage_rank_eligible"] & py_area["percentage_change"].notna()
py_area.loc[eligible_mask, "percentage_change_rank"] = (
    py_area.loc[eligible_mask]
    .groupby("current_year")["percentage_change"]
    .rank(method="min", ascending=False)
)
py_area.loc[eligible_mask, "percentage_decrease_rank"] = (
    py_area.loc[eligible_mask]
    .groupby("current_year")["percentage_change"]
    .rank(method="min", ascending=True)
)

comparison_columns = [
    "community_area", "current_year", "previous_year",
    "current_year_incident_count", "previous_year_incident_count",
    "absolute_change", "percentage_change", "percentage_rank_eligible",
    "percentage_change_rank", "percentage_decrease_rank",
    "absolute_change_rank", "absolute_decrease_rank"
]
area_validation = py_area[comparison_columns].merge(
    sql_community[comparison_columns],
    on=["community_area", "current_year"],
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)

value_columns = [
    "previous_year", "current_year_incident_count", "previous_year_incident_count",
    "absolute_change", "percentage_change", "percentage_change_rank",
    "percentage_decrease_rank", "absolute_change_rank", "absolute_decrease_rank"
]
for column in value_columns:
    area_validation[f"{column}_python"] = pd.to_numeric(
        area_validation[f"{column}_python"], errors="coerce"
    )
    area_validation[f"{column}_sql"] = pd.to_numeric(
        area_validation[f"{column}_sql"], errors="coerce"
    )

area_differences = pd.DataFrame({
    "metric": value_columns,
    "max_absolute_difference": [
        np.nanmax(np.abs(
            area_validation[f"{column}_python"].fillna(-1).to_numpy(dtype=float)
            - area_validation[f"{column}_sql"].fillna(-1).to_numpy(dtype=float)
        ))
        for column in value_columns
    ],
})
display(area_differences)

,metric,max_absolute_difference
0,previous_year,0.0000
1,current_year_incident_count,0.0000
2,previous_year_incident_count,0.0000
3,absolute_change,0.0000
4,percentage_change,0.0000
5,percentage_change_rank,0.0000
6,percentage_decrease_rank,0.0000
7,absolute_change_rank,0.0000
8,absolute_decrease_rank,0.0000


In [7]:
forest_glen = py_area.loc[
    (py_area["community_area"].eq(12)) & (py_area["current_year"].eq(2025)),
    [
        "community_area_name", "previous_year", "current_year",
        "previous_year_incident_count", "current_year_incident_count",
        "absolute_change", "percentage_change", "percentage_decrease_rank",
        "absolute_decrease_rank"
    ],
]
austin = py_area.loc[
    (py_area["community_area"].eq(25)) & (py_area["current_year"].eq(2025)),
    [
        "community_area_name", "previous_year", "current_year",
        "previous_year_incident_count", "current_year_incident_count",
        "absolute_change", "percentage_change", "percentage_decrease_rank",
        "absolute_decrease_rank"
    ],
]
display(pd.concat([forest_glen, austin], ignore_index=True).round(4))

,community_area_name,previous_year,current_year,previous_year_incident_count,current_year_incident_count,absolute_change,percentage_change,percentage_decrease_rank,absolute_decrease_rank
0,FOREST GLEN,2024,2025,545.0000,409,-136.0000,-24.9541,1.0000,46.0000
1,AUSTIN,2024,2025,"12,958.0000",11806,"-1,152.0000",-8.8903,34.0000,1.0000


## 5. Validation gate

Every check must be true. An exception stops notebook execution if any
SQL/Python discrepancy exceeds the stated numeric tolerance.

In [8]:
annual_pct_match = np.allclose(
    annual_validation["python_percentage_change"].dropna().to_numpy(dtype=float),
    annual_validation["percentage_change"].dropna().to_numpy(dtype=float),
    rtol=0,
    atol=1e-10,
)
area_value_match = all(
    np.allclose(
        area_validation[f"{column}_python"].fillna(-1).to_numpy(dtype=float),
        area_validation[f"{column}_sql"].fillna(-1).to_numpy(dtype=float),
        rtol=0,
        atol=1e-10,
    )
    for column in value_columns
)
area_eligibility_match = (
    area_validation["percentage_rank_eligible_python"].astype(bool).to_numpy()
    == area_validation["percentage_rank_eligible_sql"].astype(bool).to_numpy()
).all()

checks = {
    "Executive count/date/coverage values match": bool(executive_validation["matches"].all()),
    "Source IDs are unique in Pandas": records["source_id"].nunique() == len(records),
    "Annual counts match exactly": annual_validation["count_difference"].fillna(0).eq(0).all(),
    "Annual YoY percentages match within 1e-10": bool(annual_pct_match),
    "Community comparison row count is 154": len(area_validation) == 154,
    "Each comparison year contains 77 areas": py_area.groupby("current_year").size().eq(77).all(),
    "Community values and ranks match within 1e-10": bool(area_value_match),
    "Community percentage-rank eligibility matches": bool(area_eligibility_match),
    "Forest Glen 2025 count pair matches 545 to 409": (
        int(forest_glen.iloc[0]["previous_year_incident_count"]) == 545
        and int(forest_glen.iloc[0]["current_year_incident_count"]) == 409
    ),
    "Austin is 2025 absolute-decrease rank 1": int(austin.iloc[0]["absolute_decrease_rank"]) == 1,
}
validation_summary = pd.DataFrame(
    {"check": checks.keys(), "passed": checks.values()}
)
display(validation_summary)

failed_checks = validation_summary.loc[~validation_summary["passed"], "check"].tolist()
if failed_checks:
    raise AssertionError(f"Validation failed: {failed_checks}")

print(f"PASS: all {len(validation_summary)} SQL/Python validation checks succeeded.")
print("Maximum annual count difference: 0")
print(f"Maximum community-area numeric difference: {area_differences['max_absolute_difference'].max():.12f}")

,check,passed
0,Executive count/date/coverage values match,True
1,Source IDs are unique in Pandas,True
2,Annual counts match exactly,True
3,Annual YoY percentages match within 1e-10,True
4,Community comparison row count is 154,True
5,Each comparison year contains 77 areas,True
6,Community values and ranks match within 1e-10,True
7,Community percentage-rank eligibility matches,True
8,Forest Glen 2025 count pair matches 545 to 409,True
9,Austin is 2025 absolute-decrease rank 1,True


PASS: all 10 SQL/Python validation checks succeeded.
Maximum annual count difference: 0
Maximum community-area numeric difference: 0.000000000000


## Conclusion

A successful final cell demonstrates that Pandas independently reproduced
the Milestone 6 citywide and community-area results from record-level
data. It does not prove the source is complete or causal; it confirms
implementation consistency for the documented extract and definitions.

In [9]:
engine.dispose()
print("Database engine disposed.")

Database engine disposed.
